# Ingestion Pipeline Walkthrough

Debug/audit notebook that walks through the full ingestion pipeline for a chosen species.
Shows at each step what data the LLM received (prompts) and what it produced (extracted features).

**Cells 1–6** read from DB and cache only. **Cell 7** makes a live LLM call (requires Ollama).

In [1]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import json, re, sys
from pathlib import Path
from IPython.display import display, Markdown, HTML
import pandas as pd

# Project root on sys.path so we can import project modules
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from db.connection import get_session
from db.models import SourceObservation
from ingestion.extract import (
    SYSTEM_PROMPT_PASS1,
    SYSTEM_PROMPT_GROUP_A,
    SYSTEM_PROMPT_GROUP_B,
    SYSTEM_PROMPT_GROUP_C,
    SYSTEM_PROMPT_GROUP_D,
    SYSTEM_PROMPT_GROUP_E,
    _pass1_context,
    _MAX_TEXT_CHARS,
)
from ingestion.rubric import (
    MORPHOLOGICAL_FIELDS,
    ECOLOGICAL_FIELDS,
    TAXONOMIC_FIELDS,
    _get_nested,
)
from llm.schemas import Pass1IdentityFeatures

CACHE_DIR = ROOT / "data" / "cache"

# Source name → cache file prefix (Wikipedia has no prefix)
SOURCE_PREFIX = {
    "Wikipedia": "",
    "first-nature": "first-nature_",
    "funghiitaliani": "funghiitaliani_",
    "ultimate-mushroom": "ultimatemushroom_",
    "mushroomexpert": "mushroomexpert_",
}

# ── Pick your species ──────────────────────────────────────────────────────
SPECIES = "Amanita muscaria"  # <-- change this

safe_name = SPECIES.replace(" ", "_")

# List available cache files for this species
cache_files = sorted(CACHE_DIR.glob(f"*{safe_name}.json"))
cache_files = [f for f in cache_files if not f.name.endswith("_index.json")]
print(f"Species: {SPECIES}")
print(f"Cache files found: {len(cache_files)}")
for f in cache_files:
    print(f"  {f.name}")

Species: Amanita muscaria
Cache files found: 5
  Amanita_muscaria.json
  first-nature_Amanita_muscaria.json
  funghiitaliani_Amanita_muscaria.json
  mushroomexpert_Amanita_muscaria.json
  ultimatemushroom_Amanita_muscaria.json


In [2]:
# ── Cell 2: Cached source texts ───────────────────────────────────────────
# Load and display raw cached JSON files — what the LLM actually sees

SOURCE = "mushroomexpert"  # <-- change to inspect a different source
# 'Wikipedia' 'funghiitaliani' 'ultimate-mushroom' 'mushroomexpert'

prefix = SOURCE_PREFIX.get(SOURCE, "")
cache_path = CACHE_DIR / f"{prefix}{safe_name}.json"

if not cache_path.exists():
    print(f"Cache file not found: {cache_path}")
    print(f"Available sources: {[f.name for f in cache_files]}")
else:
    with open(cache_path) as f:
        cached = json.load(f)
    source_text = cached["text"]
    source_url = cached.get("url", "N/A")
    truncated_text = source_text[:_MAX_TEXT_CHARS]

    print(f"Source: {SOURCE}")
    print(f"URL: {source_url}")
    print(f"Full text length: {len(source_text):,} chars")
    print(f"Truncated to: {len(truncated_text):,} chars (LLM sees this)")
    print(f"\n{'='*80}")
    print(f"TRUNCATED SOURCE TEXT (first {_MAX_TEXT_CHARS} chars):")
    print(f"{'='*80}\n")
    print(truncated_text)

Source: mushroomexpert
URL: https://www.mushroomexpert.com/amanita_muscaria_flavivolvata.html
Full text length: 4,186 chars
Truncated to: 4,186 chars (LLM sees this)

TRUNCATED SOURCE TEXT (first 20000 chars):

Amanita muscaria var. flavivolvata

[ Basidiomycetes > Agaricales > Amanitaceae > Amanita . . . ]

by Michael Kuo

This is our continent's version of the classic Eurasian "toadstool," Amanita muscaria , which is probably the most depicted and recognized mushroom on earth--a fact in evidence even by my spelling checker's lack of objection to the species name; this almost never happens with fungi. To me, Amanita muscaria looks like those cut-out lawn decorations featuring an old farm woman in a polka-dotted dress bending over to tend flowers. Whatever it looks like to you, you'll probably agree that it is gorgeous. Our North American variety (or "subspecies," if you prefer) is distinguished by the fact that its universal veil and warts are yellow at first, though they quickly fade

In [3]:
# ── Cell 3: Pass 1 — prompt & result ─────────────────────────────────────
# Reconstruct the exact Pass 1 prompt, then show what the LLM produced (from DB)

# Reconstruct the user prompt (same logic as extract_features_from_text)
pass1_prompt = (
    f"Extract identity, taxonomy, body plan, and safety features "
    f"for '{SPECIES}' from the following text.\n"
    f"Source: {SOURCE}\n\n"
    f"{truncated_text}"
)

print("SYSTEM PROMPT (Pass 1):")
print("=" * 80)
print(SYSTEM_PROMPT_PASS1)
print("\n" + "=" * 80)
print("\nUSER PROMPT (Pass 1):")
print("=" * 80)
# Show just the header, not the full source text again
header_end = pass1_prompt.find(truncated_text[:50])
print(pass1_prompt[:header_end] + "[... source text as shown above ...]")

# Load Pass 1 result from DB
print("\n" + "=" * 80)
print("\nPASS 1 RESULT (from source_observations):")
print("=" * 80)

session = get_session()
obs = (
    session.query(SourceObservation)
    .filter_by(scientific_name=SPECIES, source_name=SOURCE)
    .first()
)
session.close()

if not obs:
    print(f"No source_observation found for ({SPECIES}, {SOURCE})")
else:
    feat = obs.features_json
    pass1_fields = [
        "scientific_name", "species_epithet", "common_names", "synonyms",
        "kingdom", "phylum", "order", "family", "genus",
        "overall_body_form", "overall_size_class", "growth_habit",
        "hymenium", "edibility_status", "known_toxins",
        "known_lookalikes", "extraction_notes",
    ]
    for field in pass1_fields:
        val = feat.get(field)
        if val is not None and val != [] and val != "":
            print(f"  {field}: {val}")

SYSTEM PROMPT (Pass 1):
You are a mycologist extracting structured features from species descriptions.

GENERAL RULES
- Extract ONLY what is explicitly stated in the source text. Never guess or infer.
- Use null for any feature not mentioned — omission is better than a wrong value.
- Record ambiguity or conflicting information in extraction_notes.
- scientific_name must match the queried species name exactly.
- For list fields, include every value mentioned in the text.

LANGUAGE
- ALL values MUST be in English — translate any Italian, French, German, or other
  foreign-language terms before writing them.
- Color examples: giallo → yellow, rosso → red, bruno → brown, nero → black,
  bianco/biancastro/biancastra → white/whitish, verde → green, arancione → orange,
  vinoso → wine-colored, lilacino/lilla → lilac, ocra → ochre, grigio → grey,
  rosa → pink, viola → violet, beige → beige, crema → cream.
- Compound color forms must also be translated: "vinoso-bruno" → "wine-brown",
  "rosa-l

In [4]:
# ── Cell 4: Pass 2 — prompts & results ───────────────────────────────────
# Show each group's user prompt (with pass1 context) and system prompt,
# then the corresponding extracted features from DB.

if not obs:
    print("No observation loaded — run Cell 3 first.")
else:
    feat = obs.features_json

    # Reconstruct pass1 context from stored features
    p1 = Pass1IdentityFeatures(
        scientific_name=feat.get("scientific_name", SPECIES),
        genus=feat.get("genus"),
        family=feat.get("family"),
        overall_body_form=feat.get("overall_body_form"),
        hymenium_type=(_get_nested(feat, "hymenium.type")),
        growth_habit=feat.get("growth_habit"),
    )
    context = _pass1_context(p1)

    # Conditional hint for Group B
    ht = (p1.hymenium_type or "").lower()
    if ht == "pores":
        group_b_hint = "Focus on pore and tube fields (gills are not applicable)."
    elif ht == "gills":
        group_b_hint = "Focus on gill fields (pores and tubes are not applicable)."
    else:
        group_b_hint = ""

    groups = [
        ("Group A: Cap & Surface", SYSTEM_PROMPT_GROUP_A, "cap & surface", "",
         ["cap"]),
        ("Group B: Hymenium Detail", SYSTEM_PROMPT_GROUP_B, "hymenium detail", group_b_hint,
         ["gills", "pores", "tubes", "spore_print_color"]),
        ("Group C: Stem, Veil & Volva", SYSTEM_PROMPT_GROUP_C, "stem, veil & volva", "",
         ["stem", "veil", "volva"]),
        ("Group D: Flesh & Chemistry", SYSTEM_PROMPT_GROUP_D, "flesh & chemistry", "",
         ["flesh", "chemical"]),
        ("Group E: Spore, Micro & Ecology", SYSTEM_PROMPT_GROUP_E, "spore, microscopic & ecology", "",
         ["spore", "microscopic", "ecology"]),
    ]

    for title, sys_prompt, group_label, hint, feat_keys in groups:
        hint_line = f"\n{hint}" if hint else ""
        user_prompt = (
            f"Extract {group_label} features for '{SPECIES}' "
            f"from the following text.\n"
            f"Source: {SOURCE}\n\n"
            f"CONTEXT FROM PASS 1:\n{context}\n{hint_line}\n"
            f"SOURCE TEXT:\n[... {len(truncated_text):,} chars ...]"
        )

        print(f"\n{'#'*80}")
        print(f"# {title}")
        print(f"{'#'*80}")
        print(f"\nSYSTEM PROMPT ({title}):")
        print("-" * 40)
        print(sys_prompt)
        print(f"\nUSER PROMPT ({title}):")
        print("-" * 40)
        print(user_prompt)
        print(f"\nEXTRACTED FEATURES ({title}):")
        print("-" * 40)
        for key in feat_keys:
            val = feat.get(key)
            if val is not None:
                if isinstance(val, dict):
                    for k, v in val.items():
                        if v is not None and v != [] and v != "":
                            print(f"  {key}.{k}: {v}")
                else:
                    print(f"  {key}: {val}")


################################################################################
# Group A: Cap & Surface
################################################################################

SYSTEM PROMPT (Group A: Cap & Surface):
----------------------------------------
You are a mycologist extracting structured features from species descriptions.

GENERAL RULES
- Extract ONLY what is explicitly stated in the source text. Never guess or infer.
- Use null for any feature not mentioned — omission is better than a wrong value.
- Record ambiguity or conflicting information in extraction_notes.
- scientific_name must match the queried species name exactly.
- For list fields, include every value mentioned in the text.

LANGUAGE
- ALL values MUST be in English — translate any Italian, French, German, or other
  foreign-language terms before writing them.
- Color examples: giallo → yellow, rosso → red, bruno → brown, nero → black,
  bianco/biancastro/biancastra → white/whitish, verde → green, a

In [5]:
# ── Cell 5: Feature Inspector ────────────────────────────────────────────
# Pick specific feature dot-paths, see the extracted value, which rubric
# group it belongs to, and keyword search highlighting in source text.

INSPECT_PATHS = [  # <-- change these
    "cap.shape",
    "gills.attachment",
    "ecology.trophic_mode",
    "flesh.odor",
    "spore.ornamentation",
]

if not obs:
    print("No observation loaded — run Cell 3 first.")
else:
    feat = obs.features_json

    def rubric_group(path):
        if path in MORPHOLOGICAL_FIELDS:
            return "MORPHOLOGICAL"
        if path in ECOLOGICAL_FIELDS:
            return "ECOLOGICAL"
        if path in TAXONOMIC_FIELDS:
            return "TAXONOMIC"
        return "(not in rubric)"

    def search_source(text, value):
        """Find lines in source text that mention the extracted value."""
        if value is None:
            return []
        # Build search terms from the value
        if isinstance(value, list):
            terms = [str(v).lower() for v in value if v]
        elif isinstance(value, bool):
            return []  # booleans don't map to text keywords
        else:
            terms = [str(value).lower()]

        hits = []
        for i, line in enumerate(text.split("\n"), 1):
            line_lower = line.lower()
            for term in terms:
                if term in line_lower:
                    hits.append((i, line.strip(), term))
                    break
        return hits

    for path in INSPECT_PATHS:
        value = _get_nested(feat, path)
        group = rubric_group(path)
        print(f"\n{'─'*60}")
        print(f"  Path:  {path}")
        print(f"  Value: {value}")
        print(f"  Group: {group}")

        hits = search_source(source_text, value)
        if hits:
            print(f"  Source mentions ({len(hits)} lines):")
            for lineno, line, term in hits[:5]:
                # Bold the matching term
                highlighted = line.replace(
                    term, f"**{term}**"
                ).replace(
                    term.capitalize(), f"**{term.capitalize()}**"
                )
                print(f"    L{lineno}: {highlighted[:200]}")
        else:
            if value is not None:
                print(f"  (no matching text found in source — possible hallucination?)")
            else:
                print(f"  (null — not extracted)")


────────────────────────────────────────────────────────────
  Path:  cap.shape
  Value: convex
  Group: MORPHOLOGICAL
  Source mentions (1 lines):
    L19: Cap: 5-25 cm; nearly oval or round at first, becoming **convex**, then broadly **convex** to flat in age; bald; adorned with numerous small, cottony warts that are initially yellow but very quickly fa

────────────────────────────────────────────────────────────
  Path:  gills.attachment
  Value: adnate
  Group: MORPHOLOGICAL
  (no matching text found in source — possible hallucination?)

────────────────────────────────────────────────────────────
  Path:  ecology.trophic_mode
  Value: mycorrhizal
  Group: ECOLOGICAL
  Source mentions (1 lines):
    L17: Ecology: **Mycorrhizal** with conifers and hardwoods (primarily oaks); summer and fall (and over winter in coastal California); fairly widely distributed in North America, but most common in the Pacif

────────────────────────────────────────────────────────────
  Path:  flesh.od

In [6]:
# ── Cell 6: Cross-source comparison ──────────────────────────────────────
# If multiple sources exist, show a side-by-side DataFrame of selected
# features across all source_observations, plus the reconciled values.

from db.models import ReconciledSpecies

COMPARE_FIELDS = [  # <-- change these
    "cap.shape",
    "cap.colors",
    "cap.surface_texture",
    "hymenium.type",
    "gills.attachment",
    "gills.spacing",
    "stem.shape",
    "stem.hollow_or_solid",
    "flesh.odor",
    "flesh.taste",
    "ecology.trophic_mode",
    "ecology.associated_trees",
    "edibility_status",
    "overall_body_form",
]

session = get_session()
all_obs = (
    session.query(SourceObservation)
    .filter_by(scientific_name=SPECIES)
    .all()
)
reconciled = (
    session.query(ReconciledSpecies)
    .filter_by(scientific_name=SPECIES)
    .first()
)
session.close()

if len(all_obs) < 2:
    print(f"Only {len(all_obs)} source(s) for {SPECIES} — need 2+ for comparison.")
    if all_obs:
        print(f"  Available: {[o.source_name for o in all_obs]}")
else:
    rows = {}
    for o in all_obs:
        col = {}
        for path in COMPARE_FIELDS:
            val = _get_nested(o.features_json, path)
            if isinstance(val, list):
                col[path] = ", ".join(str(v) for v in val)
            else:
                col[path] = val
        rows[o.source_name] = col

    # Add reconciled column if available
    if reconciled:
        rec_col = {}
        for path in COMPARE_FIELDS:
            val = _get_nested(reconciled.features_json, path)
            rec_col[path] = ", ".join(str(v) for v in val) if isinstance(val, list) else val
        rows["RECONCILED"] = rec_col

    df = pd.DataFrame(rows)
    print(f"Cross-source comparison for {SPECIES} ({len(all_obs)} sources):\n")
    with pd.option_context("display.max_colwidth", 60, "display.max_rows", None):
        display(df)

    # Reconciliation metadata
    if reconciled:
        print(f"\n{'─'*60}")
        print("Reconciliation metadata:")
        print(f"  confidence:   {reconciled.reconciliation_confidence}")
        print(f"  needs_review: {reconciled.needs_review}")
        print(f"  review_notes: {reconciled.review_notes}")
        print(f"  source_count: {reconciled.source_count}")
        print(f"  reconciled_at: {reconciled.reconciled_at}")
    else:
        print(f"\nNo reconciled row found for {SPECIES}.")

2026-03-03 08:20:09,826 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 08:20:09,829 INFO sqlalchemy.engine.Engine SELECT source_observations.id AS source_observations_id, source_observations.scientific_name AS source_observations_scientific_name, source_observations.source_name AS source_observations_source_name, source_observations.source_url AS source_observations_source_url, source_observations.source_text_hash AS source_observations_source_text_hash, source_observations.features_json AS source_observations_features_json, source_observations.extraction_model AS source_observations_extraction_model, source_observations.extraction_timestamp AS source_observations_extraction_timestamp, source_observations.extraction_notes AS source_observations_extraction_notes, source_observations.species_id AS source_observations_species_id 
FROM source_observations 
WHERE source_observations.scientific_name = %(scientific_name_1)s
2026-03-03 08:20:09,830 INFO sqlalchemy.engine.Engine [gen

,Wikipedia,first-nature,mushroomexpert,ultimate-mushroom,funghiitaliani,RECONCILED
cap.shape,globose,convex,convex,convex,convex,convex
cap.colors,"red, white","red, orange, yellow",red,"red, brick-red, yellow-red, red-orange","red, red-orange, dark red","red, orange, yellow, brick-red, red-orange, dark red"
cap.surface_texture,warty,smooth,smooth,scaly,warty,warty
hymenium.type,gills,gills,gills,gills,gills,gills
gills.attachment,free,free,adnate,free,free,free
gills.spacing,None,crowded,close,crowded,distant,crowded
stem.shape,equal,equal,equal,cylindrical,equal,equal
stem.hollow_or_solid,solid,None,None,hollow,hollow,hollow
flesh.odor,mild,not distinctive,none,none,none,none
flesh.taste,None,None,None,sweet,sweet,sweet



────────────────────────────────────────────────────────────
Reconciliation metadata:
  confidence:   0.95
  needs_review: True
  review_notes: Conflicting information on cap fading, stem consistency, bruising color, and spore shape.  The presence/absence of bruising on the stem is inconsistent. Spore shape varies between sources. Cap fading is reported as yellow or pale orange. Stem consistency is reported as brittle, fibrous, or soft.
  source_count: 5
  reconciled_at: 2026-03-02 20:58:35.463744


In [7]:
# ── Cell 7: Live reconciliation — prompt & LLM response ─────────────────
# Makes a live LLM call using the same pipeline as ingestion/reconcile.py.
# Shows the system prompt, user prompt, and full structured response.

from ingestion.reconcile import SYSTEM_PROMPT as RECONCILE_SYSTEM_PROMPT
from llm.client import structured_completion
from llm.schemas import ReconciliationResult

if len(all_obs) < 2:
    print(f"Only {len(all_obs)} source(s) — reconciliation needs 2+.")
else:
    # Build prompt exactly like reconcile_species() does
    sources_text = "\n\n".join(
        f"Source {i + 1} ({obs.source_name}):\n"
        + json.dumps(obs.features_json, ensure_ascii=False, separators=(",", ":"))
        for i, obs in enumerate(all_obs)
    )
    prompt = (
        f"Reconcile the following {len(all_obs)} source descriptions "
        f"for '{SPECIES}':\n\n{sources_text}"
    )

    # Show prompts
    print("SYSTEM PROMPT:")
    print("=" * 80)
    print(RECONCILE_SYSTEM_PROMPT)
    print("\n" + "=" * 80)
    print("\nUSER PROMPT:")
    print("=" * 80)
    print(prompt[:2000] + "\n[... truncated for display ...]" if len(prompt) > 2000 else prompt)

    # Live LLM call
    print("\n" + "=" * 80)
    print("Calling LLM for reconciliation...")
    result = structured_completion(
        prompt=prompt,
        response_model=ReconciliationResult,
        system=RECONCILE_SYSTEM_PROMPT,
        temperature=0.1,
        max_tokens=4096,
    )

    # Display response
    print("\nRESULT:")
    print("=" * 80)
    print(f"  confidence:   {result.confidence}")
    print(f"  conflicts:    {result.conflicts}")
    print(f"  needs_review: {result.needs_review}")
    print(f"  review_notes: {result.review_notes}")
    print(f"\nReconciled features:")
    print(json.dumps(result.reconciled_features.model_dump(), indent=2, ensure_ascii=False))

SYSTEM PROMPT:
You are a mycologist reconciling multiple source descriptions of the same mushroom species.

Rules:
- Merge complementary information from different sources (prefer more specific values).
- For conflicts: choose the most specific, well-supported value and note the conflict.
- Never guess or infer — only use what is explicitly stated in the sources.
- Set needs_review=True if any critical safety fields (edibility, known_toxins) conflict.
- Preserve all common_names, known_toxins, known_lookalikes from all sources.
- scientific_name must match the queried species exactly.



USER PROMPT:
Reconcile the following 5 source descriptions for 'Amanita muscaria':

Source 1 (Wikipedia):
{"cap":{"shape":"globose","colors":["red","white"],"color_faded":null,"margin_type":null,"color_pattern":"mottled","bruising_color":null,"diameter_max_cm":30.0,"diameter_min_cm":5.0,"scales_or_warts":"white to yellow pyramid-shaped warts","surface_texture":"warty","surface_moisture":null,"central_d